# v2 — natural-language parsing layer, exploration

Consumer notebook for `tariffs.nlp.parse_vessel_request()`. Kept separate from `notebooks/exploration.ipynb` on purpose: this one needs `ANTHROPIC_API_KEY` and makes real (non-deterministic) model calls, the v1 notebook doesn't and shouldn't have to.

Launch with the key in the kernel's environment: `uv run --env-file .env jupyter lab`.

No formulas, rate values, or extraction logic defined here — it only calls the package. `parse_vessel_request()` returns a `Parsed` or a `Rejected`; a bad request is a normal result to inspect, not an exception.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from tariffs.nlp import parse_vessel_request, Parsed, Rejected
from tariffs.engine import calculate

## Parse a free-text request

In [3]:
request_text = (
    "The bulk carrier SUDESTADA (Malta flag, built 2010) called at the Port of Durban. "
    "GT 51,300, LOA 229.2m. She arrived 15 Nov 2024 10:12 and departed 22 Nov 2024 13:00. "
    "Days alongside: 3.396. Number of Operations: 2. She was exporting 40,000 MT of iron ore. "
    "No mooring boat was used."
)

parsed = parse_vessel_request(request_text)
assert isinstance(parsed, Parsed), parsed  # would show parsed.reason if this were Rejected
parsed.call

VesselCall(vessel_name='SUDESTADA', port=<Port.DURBAN: 'durban'>, gross_tonnage=51300.0, length_overall_m=229.2, vessel_type=<VesselType.BULK_CARRIER: 'bulk_carrier'>, arrival=datetime.datetime(2024, 11, 15, 10, 12), departure=datetime.datetime(2024, 11, 22, 13, 0), chargeable_period_days=3.396, chargeable_period_basis=<PeriodBasis.DAYS_ALONGSIDE_PROXY: 'days_alongside_proxy'>, number_of_operations=2, marine_service_count=None, engaged_in_cargo_working=None, is_bona_fide_coaster=None, is_passenger_vessel=None, is_first_sa_port_call=None, days_in_sa_waters=None, call_purpose_bunkers_stores_water_only=None, hull_certification=[], exemption_status=None, self_propelled=None, mooring_boat_used=False, additional_tug_requested=None, vessel_without_own_power=None, service_cancelled_after_standby=None, late_against_notified_time=None)

## What was extracted, and from where

One row per field the model actually populated, with the verbatim text fragment it came from. Fields not shown here were left `None` — never guessed.

In [4]:
parsed.trace_df()

,field,value,evidence
0,vessel_name,SUDESTADA,SUDESTADA
1,port,Port.DURBAN,Port of Durban
2,gross_tonnage,51300.0,"GT 51,300"
3,length_overall_m,229.2,LOA 229.2m
4,vessel_type,VesselType.BULK_CARRIER,bulk carrier
5,arrival,2024-11-15 10:12:00,arrived 15 Nov 2024 10:12
6,departure,2024-11-22 13:00:00,departed 22 Nov 2024 13:00
7,chargeable_period_days,3.396,Days alongside: 3.396
8,chargeable_period_basis,PeriodBasis.DAYS_ALONGSIDE_PROXY,Days alongside: 3.396
9,number_of_operations,2,Number of Operations: 2


## What the parser alone already knows is computable

`Parsed.tariffs` reports, per tariff, whether the request supported computing it — never a silent zero for one that couldn't be. This request is complete, so all six are computed.

In [5]:
parsed.totals()

{'light_dues': 60062.04,
 'vts_dues': 33345.0,
 'pilotage_dues': 47189.94,
 'towage_dues': 147074.38,
 'berthing_services': 19639.5,
 'port_dues': 199549.22}

## Feed the validated call into the unchanged v1 engine

For the full trace, modifiers, and warnings — `calculate()` doesn't know or care that its input came from an LLM.

In [6]:
result = calculate(parsed.call)
result.totals()

{'light_dues': 60062.04,
 'port_dues': 199549.22,
 'towage_dues': 147074.38,
 'vts_dues': 33345.0,
 'pilotage_dues': 47189.94,
 'berthing_services': 19639.5,
 'running_of_vessel_lines': None}

In [7]:
result.trace_df()

,tariff,section,page,description,inputs,rounding,modifier,modifier_resolution,subtotal
0,Light dues,1.1.1,9,ceil(GT/100) x rate_per_100t (foreign/other ve...,"{'gross_tonnage': 51300.0, 'units': 513, 'rate...",ceil_per_100_t,NaN,NaN,60062.04
1,Light dues,1.1.1,9,modifier: exemption,{},NaN,exemption,"unresolved, base case (no modifier applied) — ...",NaN
2,Port dues,4.1.1,21,basic = ceil(GT/100) x basic_rate_per_100t,"{'gross_tonnage': 51300.0, 'units': 513, 'basi...",pro_rata_time,NaN,NaN,98870.49
3,Port dues,4.1.1,21,incremental = ceil(GT/100) x incremental_rate_...,"{'chargeable_period_days': 3.396, 'chargeable_...",pro_rata_time,NaN,NaN,100678.73
4,Port dues,4.1.1,21,modifier: exemption,{},NaN,exemption,"unresolved, base case (no modifier applied) — ...",NaN
5,Port dues,4.1.1,21,modifier: port_dues_reduction_60pct,{},NaN,port_dues_reduction_60pct,not applied — condition not met / not stated,NaN
6,Port dues,4.1.1,21,modifier: port_dues_reduction_35pct,{},NaN,port_dues_reduction_35pct,"unresolved, base case (no modifier applied) — ...",NaN
7,Port dues,4.1.1,21,modifier: port_dues_reduction_10pct,{},NaN,port_dues_reduction_10pct,not applied — not a certified tanker / not stated,NaN
8,Port dues,4.1.1,21,modifier: port_dues_reduction_15pct,{},NaN,port_dues_reduction_15pct,not applied — stay not under 12h / not stated,NaN
9,Port dues,4.1.1,21,modifier: port_dues_long_stay_surcharge_20pct,{},NaN,port_dues_long_stay_surcharge_20pct,not applied — condition not met / not stated,NaN


## A bad request is a normal result, not an exception

In [8]:
off_topic = parse_vessel_request("What's the weather like in Cape Town today?")
isinstance(off_topic, Rejected), off_topic.reason if isinstance(off_topic, Rejected) else None

(True,
 'This tool calculates Transnet National Ports Authority (TNPA) port tariffs for a vessel call at a South African port — it can\'t help with that request. Example of what it expects: "The bulk carrier SUDESTADA, GT 51,300, called at the Port of Durban. Arrived 15 Nov 2024, departed 22 Nov 2024. Number of Operations: 2."')

In [9]:
wrong_port = parse_vessel_request("A bulk carrier, GT 40,000, called at the Port of Walvis Bay.")
isinstance(wrong_port, Rejected), wrong_port.reason if isinstance(wrong_port, Rejected) else None

(True,
 "'Walvis Bay' is not one of the eight ports this tool covers: Richards Bay, Durban, East London, Ngqura, Port Elizabeth, Mossel Bay, Cape Town, or Saldanha.")

## An incomplete request: some tariffs computed, the rest reported not computable

Missing `number_of_operations` and a chargeable period — not enough to reject the request (port and GT are present), but not enough for four of the six tariffs either. None of them silently becomes zero.

In [10]:
partial = parse_vessel_request("A bulk carrier, GT 51,255, called at the Port of Durban.")
assert isinstance(partial, Parsed)
partial.totals()

{'light_dues': 60062.04,
 'vts_dues': 33315.75,
 'pilotage_dues': None,
 'towage_dues': None,
 'berthing_services': None,
 'port_dues': None}

In [11]:
{name: outcome.reason for name, outcome in partial.tariffs.items() if not outcome.computed}

{'pilotage_dues': 'not computable — missing number_of_operations',
 'towage_dues': 'not computable — missing number_of_operations',
 'berthing_services': 'not computable — missing number_of_operations',
 'port_dues': 'not computable — missing chargeable_period_days'}